In [1]:
import json
import glob
import os
from pathlib import Path
import pandas as pd

# ==========================================
# 1. 基础路径配置
# ==========================================
BASE_DIR = Path('/Volumes/KIOXIA/Project/lob')
# 专门建一个干净的文件夹存纯交易数据
OUTPUT_DIR = BASE_DIR / 'raw_transactions_only'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEEKS_TO_PROCESS = ['week1', 'week2']

# ==========================================
# 2. 核心提取函数：只抓取真正的成交记录
# ==========================================
def extract_raw_transactions(jsonl_path):
    transactions = []
    
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            
            data = json.loads(line)
            records = data if isinstance(data, list) else [data]
            
            for rec in records:
                # 【核心过滤逻辑】：根据你的真实文件结构，真正的交易包含 transaction_hash
                if rec.get('event_type') == 'last_trade_price' or 'transaction_hash' in rec:
                    
                    # 提取你要的所有交易相关字段
                    transactions.append({
                        'timestamp': pd.to_datetime(int(rec.get('timestamp')), unit='ms', utc=True),
                        'market': rec.get('market', ''),
                        'asset_id': rec.get('asset_id', ''),
                        'side': rec.get('side', ''),
                        'price': float(rec.get('price')) if rec.get('price') else None,
                        'size': float(rec.get('size')) if rec.get('size') else None,
                        'fee_rate_bps': rec.get('fee_rate_bps', ''),
                        'transaction_hash': rec.get('transaction_hash', '')
                    })
                    
    df_tx = pd.DataFrame(transactions)
    
    # 按照时间排个序
    if not df_tx.empty:
        df_tx.sort_values('timestamp', inplace=True)
        df_tx.reset_index(drop=True, inplace=True)
        
    return df_tx

# ==========================================
# 3. 遍历你指定的全部路径
# ==========================================
for week in WEEKS_TO_PROCESS:
    print(f"\n🚀 开始提取 {week} 的交易数据...")
    
    # 构建输入和输出的完整路径
    CONTRACTS_DIR = BASE_DIR / week / 'contracts' / 'btc'
    WEEK_OUTPUT_DIR = OUTPUT_DIR / week
    WEEK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # 获取该路径下所有的 jsonl 文件
    jsonl_files = glob.glob(str(CONTRACTS_DIR / '*.jsonl'))
    print(f"   找到 {len(jsonl_files)} 个合约文件，正在处理...")
    
    for file_path in jsonl_files:
        slug = os.path.basename(file_path).replace('.jsonl', '')
        
        # 调用提取函数
        df_contract_txs = extract_raw_transactions(file_path)
        
        # 如果这个文件里有交易数据，就保存下来
        if df_contract_txs is not None and not df_contract_txs.empty:
            out_file = WEEK_OUTPUT_DIR / f"{slug}_raw_txs.parquet"
            df_contract_txs.to_parquet(out_file)
            print(f"     [√] 保存成功 -> {out_file.name} (共 {len(df_contract_txs)} 条交易记录)")
        else:
            print(f"     [!] 跳过 {slug}：未发现任何交易数据")

print("\n🎉 所有路径的数据提取完毕！")


🚀 开始提取 week1 的交易数据...
   找到 627 个合约文件，正在处理...
     [√] 保存成功 -> btc-updown-15m-1765153800_raw_txs.parquet (共 1781 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1764838800_raw_txs.parquet (共 1608 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1764656100_raw_txs.parquet (共 1917 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1765081800_raw_txs.parquet (共 1707 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1764792900_raw_txs.parquet (共 3153 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1764834300_raw_txs.parquet (共 1946 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1765107900_raw_txs.parquet (共 1788 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1764567900_raw_txs.parquet (共 773 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1764745200_raw_txs.parquet (共 1657 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1764603000_raw_txs.parquet (共 3247 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1765127700_raw_txs.parquet (共 1655 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1765084500_raw_txs.parquet (共 2165 条交易记录)
     [√] 保存成功 -> btc-updown-15m-1764810000_raw_txs.parquet (共 1141

In [3]:
import pandas as pd
from pathlib import Path
import glob
import os

# ==========================================
# 1. 路径配置 (对接你刚才存好的数据)
# ==========================================
BASE_DIR = Path('/Volumes/KIOXIA/Project/lob')
INPUT_DIR = BASE_DIR / 'raw_transactions_only'  # 第一步输出的纯交易数据目录
OUTPUT_DIR = BASE_DIR / 'final_aggregated_trades' # 最终聚合好的特征存放目录

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEEKS_TO_PROCESS = ['week1', 'week2']

# ==========================================
# 2. 核心聚合函数 (完美平替你之前的 dict 逻辑)
# ==========================================
def aggregate_trades_to_second(df_raw, slug):
    if df_raw.empty:
        return pd.DataFrame()

    # 1. 把时间向下取整到秒 (例如 10:00:01.345 -> 10:00:01)
    df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
    
    # 2. 计算每笔交易的名义价值 (price * size)
    df_raw['notional'] = df_raw['price'] * df_raw['size']
    df_raw['slug'] = slug  # 补充 slug 列
    
    key_cols = ["slug", "market", "asset_id", "timestamp"]
    
    # 3. 分离 BUY 和 SELL
    buy_df = df_raw[df_raw['side'] == 'BUY']
    sell_df = df_raw[df_raw['side'] == 'SELL']
    
    # 4. 分别对 BUY 进行秒级聚合
    buy_agg = pd.DataFrame()
    if not buy_df.empty:
        buy_agg = buy_df.groupby(key_cols).agg(
            buy_trade_size=('size', 'sum'),
            buy_trade_count=('transaction_hash', 'count'),
            taker_buy_trade_nominal_value=('notional', 'sum')
        )
        buy_agg['buy_trade_vwap'] = buy_agg['taker_buy_trade_nominal_value'] / buy_agg['buy_trade_size']
    
    # 5. 分别对 SELL 进行秒级聚合
    sell_agg = pd.DataFrame()
    if not sell_df.empty:
        sell_agg = sell_df.groupby(key_cols).agg(
            sell_trade_size=('size', 'sum'),
            sell_trade_count=('transaction_hash', 'count'),
            taker_sell_trade_nominal_value=('notional', 'sum')
        )
        sell_agg['sell_trade_vwap'] = sell_agg['taker_sell_trade_nominal_value'] / sell_agg['sell_trade_size']
    
    # 6. 把 BUY 和 SELL 合并在一起 (how='outer' 保证单边有交易的秒数也不丢失)
    if buy_agg.empty and sell_agg.empty:
        return pd.DataFrame()
    elif buy_agg.empty:
        result = sell_agg.reset_index()
    elif sell_agg.empty:
        result = buy_agg.reset_index()
    else:
        result = buy_agg.join(sell_agg, how='outer').reset_index()
    
    # 7. 处理缺失值：交易量和次数空缺填 0，VWAP 保持 NaN
    fill_zero_cols = [c for c in ['buy_trade_size', 'buy_trade_count', 'taker_buy_trade_nominal_value', 
                                  'sell_trade_size', 'sell_trade_count', 'taker_sell_trade_nominal_value'] 
                      if c in result.columns]
    result[fill_zero_cols] = result[fill_zero_cols].fillna(0)
    
    # 按照时间排序
    return result.sort_values(key_cols).reset_index(drop=True)

# ==========================================
# 3. 批量处理所有提取好的 Parquet 文件
# ==========================================
for week in WEEKS_TO_PROCESS:
    print(f"\n{'='*50}")
    print(f"🚀 开始聚合 {week.upper()} 的交易特征")
    print(f"{'='*50}")
    
    WEEK_INPUT_DIR = INPUT_DIR / week
    WEEK_OUTPUT_DIR = OUTPUT_DIR / week
    WEEK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # 获取该周所有的交易记录文件
    parquet_files = glob.glob(str(WEEK_INPUT_DIR / '*_raw_txs.parquet'))
    print(f"   [!] 找到 {len(parquet_files)} 个合约文件待聚合...")
    
    for file_path in parquet_files:
        slug = os.path.basename(file_path).replace('_raw_txs.parquet', '')
        
        # 读取第一步清洗好的纯交易数据
        df_raw = pd.read_parquet(file_path)
        
        # 执行聚合逻辑
        df_agg = aggregate_trades_to_second(df_raw, slug)
        
        # 保存最终的特征表
        if not df_agg.empty:
            out_file = WEEK_OUTPUT_DIR / f"{slug}_trade_features.parquet"
            df_agg.to_parquet(out_file)
            print(f"     [√] 聚合完成 -> {out_file.name} (输出行数: {len(df_agg)} 秒)")
        else:
            print(f"     [!] 跳过 {slug}：聚合后无有效数据")

print("\n🎉 恭喜！所有交易特征聚合完毕！")


🚀 开始聚合 WEEK1 的交易特征
   [!] 找到 627 个合约文件待聚合...
     [√] 聚合完成 -> btc-updown-15m-1765153800_trade_features.parquet (输出行数: 809 秒)
     [√] 聚合完成 -> btc-updown-15m-1764838800_trade_features.parquet (输出行数: 720 秒)
     [√] 聚合完成 -> btc-updown-15m-1764656100_trade_features.parquet (输出行数: 918 秒)
     [√] 聚合完成 -> btc-updown-15m-1765081800_trade_features.parquet (输出行数: 791 秒)
     [√] 聚合完成 -> btc-updown-15m-1764792900_trade_features.parquet (输出行数: 1174 秒)
     [√] 聚合完成 -> btc-updown-15m-1764834300_trade_features.parquet (输出行数: 877 秒)
     [√] 聚合完成 -> btc-updown-15m-1765107900_trade_features.parquet (输出行数: 911 秒)
     [√] 聚合完成 -> btc-updown-15m-1764567900_trade_features.parquet (输出行数: 467 秒)
     [√] 聚合完成 -> btc-updown-15m-1764745200_trade_features.parquet (输出行数: 844 秒)
     [√] 聚合完成 -> btc-updown-15m-1764603000_trade_features.parquet (输出行数: 1002 秒)
     [√] 聚合完成 -> btc-updown-15m-1765127700_trade_features.parquet (输出行数: 702 秒)
     [√] 聚合完成 -> btc-updown-15m-1765084500_trade_features.parquet (输出行数:

/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000

     [√] 聚合完成 -> btc-updown-15m-1764810000_trade_features.parquet (输出行数: 600 秒)
     [√] 聚合完成 -> btc-updown-15m-1764843300_trade_features.parquet (输出行数: 962 秒)
     [√] 聚合完成 -> btc-updown-15m-1764642600_trade_features.parquet (输出行数: 794 秒)
     [√] 聚合完成 -> btc-updown-15m-1764981900_trade_features.parquet (输出行数: 279 秒)
     [√] 聚合完成 -> btc-updown-15m-1765103400_trade_features.parquet (输出行数: 803 秒)
     [√] 聚合完成 -> btc-updown-15m-1764918000_trade_features.parquet (输出行数: 1096 秒)
     [√] 聚合完成 -> btc-updown-15m-1764627300_trade_features.parquet (输出行数: 886 秒)
     [√] 聚合完成 -> btc-updown-15m-1764869400_trade_features.parquet (输出行数: 1071 秒)
     [√] 聚合完成 -> btc-updown-15m-1764675000_trade_features.parquet (输出行数: 1083 秒)
     [√] 聚合完成 -> btc-updown-15m-1764882000_trade_features.parquet (输出行数: 1183 秒)
     [√] 聚合完成 -> btc-updown-15m-1765010700_trade_features.parquet (输出行数: 950 秒)
     [√] 聚合完成 -> btc-updown-15m-1764673200_trade_features.parquet (输出行数: 1024 秒)
     [√] 聚合完成 -> btc-updown-15m-176

/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000

     [√] 聚合完成 -> btc-updown-15m-1764644400_trade_features.parquet (输出行数: 927 秒)
     [√] 聚合完成 -> btc-updown-15m-1765118700_trade_features.parquet (输出行数: 1466 秒)
     [√] 聚合完成 -> btc-updown-15m-1764845100_trade_features.parquet (输出行数: 1036 秒)
     [√] 聚合完成 -> btc-updown-15m-1764966600_trade_features.parquet (输出行数: 631 秒)
     [√] 聚合完成 -> btc-updown-15m-1764578700_trade_features.parquet (输出行数: 648 秒)
     [√] 聚合完成 -> btc-updown-15m-1765082700_trade_features.parquet (输出行数: 930 秒)
     [√] 聚合完成 -> btc-updown-15m-1764817200_trade_features.parquet (输出行数: 924 秒)
     [√] 聚合完成 -> btc-updown-15m-1764946800_trade_features.parquet (输出行数: 675 秒)
     [√] 聚合完成 -> btc-updown-15m-1765068300_trade_features.parquet (输出行数: 1068 秒)
     [√] 聚合完成 -> btc-updown-15m-1764815400_trade_features.parquet (输出行数: 960 秒)
     [√] 聚合完成 -> btc-updown-15m-1765018800_trade_features.parquet (输出行数: 1031 秒)
     [√] 聚合完成 -> btc-updown-15m-1764779400_trade_features.parquet (输出行数: 700 秒)
     [√] 聚合完成 -> btc-updown-15m-1764

...

/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000

     [√] 聚合完成 -> btc-updown-15m-1765414800_trade_features.parquet (输出行数: 990 秒)
     [√] 聚合完成 -> btc-updown-15m-1765665000_trade_features.parquet (输出行数: 685 秒)
     [√] 聚合完成 -> btc-updown-15m-1765715400_trade_features.parquet (输出行数: 894 秒)
     [√] 聚合完成 -> btc-updown-15m-1765434600_trade_features.parquet (输出行数: 553 秒)
     [√] 聚合完成 -> btc-updown-15m-1765276200_trade_features.parquet (输出行数: 1193 秒)
     [√] 聚合完成 -> btc-updown-15m-1765354500_trade_features.parquet (输出行数: 749 秒)
     [√] 聚合完成 -> btc-updown-15m-1765280700_trade_features.parquet (输出行数: 1012 秒)
     [√] 聚合完成 -> btc-updown-15m-1765593900_trade_features.parquet (输出行数: 685 秒)
     [√] 聚合完成 -> btc-updown-15m-1765486800_trade_features.parquet (输出行数: 961 秒)
     [√] 聚合完成 -> btc-updown-15m-1765730700_trade_features.parquet (输出行数: 1470 秒)
     [√] 聚合完成 -> btc-updown-15m-1765371600_trade_features.parquet (输出行数: 1270 秒)
     [√] 聚合完成 -> btc-updown-15m-1765199700_trade_features.parquet (输出行数: 1045 秒)
     [√] 聚合完成 -> btc-updown-15m-176

/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000

     [√] 聚合完成 -> btc-updown-15m-1765249200_trade_features.parquet (输出行数: 1186 秒)
     [√] 聚合完成 -> btc-updown-15m-1765646100_trade_features.parquet (输出行数: 873 秒)
     [√] 聚合完成 -> btc-updown-15m-1765325700_trade_features.parquet (输出行数: 780 秒)
     [√] 聚合完成 -> btc-updown-15m-1765421100_trade_features.parquet (输出行数: 1291 秒)
     [√] 聚合完成 -> btc-updown-15m-1765445400_trade_features.parquet (输出行数: 1115 秒)
     [√] 聚合完成 -> btc-updown-15m-1765206000_trade_features.parquet (输出行数: 1140 秒)
     [√] 聚合完成 -> btc-updown-15m-1765522800_trade_features.parquet (输出行数: 419 秒)
     [√] 聚合完成 -> btc-updown-15m-1765305900_trade_features.parquet (输出行数: 1541 秒)
     [√] 聚合完成 -> btc-updown-15m-1765313100_trade_features.parquet (输出行数: 1433 秒)

🎉 恭喜！所有交易特征聚合完毕！


/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/3235670114.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')


In [4]:
import pandas as pd
from pathlib import Path
import glob

# ==========================================
# 1. 路径配置 (读取上一步聚合好的目录)
# ==========================================
BASE_DIR = Path('/Volumes/KIOXIA/Project/lob')
INPUT_DIR = BASE_DIR / 'final_aggregated_trades'
WEEKS_TO_PROCESS = ['week1', 'week2']

all_dfs = []

print("🚀 开始合并所有聚合后的交易特征表...\n")

# ==========================================
# 2. 遍历读取并合并
# ==========================================
for week in WEEKS_TO_PROCESS:
    week_dir = INPUT_DIR / week
    
    # 查找该周下面所有的 parquet 文件
    parquet_files = glob.glob(str(week_dir / '*_trade_features.parquet'))
    
    if not parquet_files:
        print(f"⚠️ 警告：在 {week} 未找到任何特征文件！请检查上一步是否成功。")
        continue
        
    print(f"   [{week}] 找到 {len(parquet_files)} 个合约文件，正在拼接...")
    
    for f in parquet_files:
        df = pd.read_parquet(f)
        all_dfs.append(df)

# ==========================================
# 3. 生成大总表并打印核对信息
# ==========================================
if all_dfs:
    # 纵向合并所有表格
    df_grand_total = pd.concat(all_dfs, ignore_index=True)
    
    # 按照 时间 和 合约名称 重新排序，让数据规规矩矩
    print("   [!] 正在按照时间和合约名称进行全盘排序...")
    df_grand_total.sort_values(by=['timestamp', 'slug'], inplace=True)
    df_grand_total.reset_index(drop=True, inplace=True)
    
    # 保存这份无敌大总表
    output_path = INPUT_DIR / "grand_total_trades_master.parquet"
    df_grand_total.to_parquet(output_path)
    
    print("\n🎉 大总表合并大功告成！")
    print("=" * 50)
    print(f"📊 最终总表行数 (Rows): {df_grand_total.shape[0]} 行")
    print(f"📊 最终总表列数 (Cols): {df_grand_total.shape[1]} 列")
    print("=" * 50)
    print(f"💾 文件已保存至: {output_path}\n")
    
    print("👀 前 5 行数据预览:")
    print(df_grand_total.head())
else:
    print("❌ 未找到任何数据进行合并，请检查路径或上一步的输出。")

🚀 开始合并所有聚合后的交易特征表...

   [week1] 找到 627 个合约文件，正在拼接...
   [week2] 找到 628 个合约文件，正在拼接...
   [!] 正在按照时间和合约名称进行全盘排序...

🎉 大总表合并大功告成！
📊 最终总表行数 (Rows): 1179430 行
📊 最终总表列数 (Cols): 12 列
💾 文件已保存至: /Volumes/KIOXIA/Project/lob/final_aggregated_trades/grand_total_trades_master.parquet

👀 前 5 行数据预览:
                        slug  \
0  btc-updown-15m-1764565200   
1  btc-updown-15m-1764565200   
2  btc-updown-15m-1764565200   
3  btc-updown-15m-1764565200   
4  btc-updown-15m-1764565200   

                                              market  \
0  0xdc2dc0399bf6584e9b502ca51d61a8a9570b731e4e5b...   
1  0xdc2dc0399bf6584e9b502ca51d61a8a9570b731e4e5b...   
2  0xdc2dc0399bf6584e9b502ca51d61a8a9570b731e4e5b...   
3  0xdc2dc0399bf6584e9b502ca51d61a8a9570b731e4e5b...   
4  0xdc2dc0399bf6584e9b502ca51d61a8a9570b731e4e5b...   

                                            asset_id  \
0  4070657408180023518277177261998523093049036367...   
1  4070657408180023518277177261998523093049036367...   
2  762445035467

In [8]:
import pandas as pd
from pathlib import Path
import glob

# ==========================================
# 1. 路径配置
# ==========================================
BASE_DIR = Path('/Volumes/KIOXIA/Project/lob')
INPUT_PATH = BASE_DIR / 'final_aggregated_trades' / 'grand_total_trades_master.parquet'
OUTPUT_PATH = BASE_DIR / 'final_aggregated_trades' / 'grand_total_trades_WIDE_master.parquet'

print("🚀 正在加载交易总表...")
df_trades = pd.read_parquet(INPUT_PATH)

# ==========================================
# 2. 从 CSV 中提取 asset_id -> Up/Down 的映射
# ==========================================
print("🔍 正在构建 Token 映射字典...")
mapping_dict = {}
# 找到所有的 trades_week.csv
csv_files = glob.glob(str(BASE_DIR / '*' / 'trades_week*.csv'))

for f in csv_files:
    df_meta = pd.read_csv(f, dtype={'token_id': str})
    # 把 token_id 和 token_name (Up/Down) 变成字典
    temp_dict = df_meta.drop_duplicates('token_id').set_index('token_id')['token_name'].to_dict()
    mapping_dict.update(temp_dict)

# ==========================================
# 3. 翻译 asset_id 并进行透视 (Pivot)
# ==========================================
print("🔄 正在将长表横向展开为宽表 (Pivot)...")

# 翻译: 匹配出 Up 和 Down
df_trades['token_type'] = df_trades['asset_id'].map(mapping_dict)

# 踢掉没有匹配上类型的数据（安全校验）
df_trades.dropna(subset=['token_type'], inplace=True)

# 找出需要拆分为 Up/Down 的数值列
val_cols = [c for c in df_trades.columns if c not in ['slug', 'market', 'asset_id', 'timestamp', 'token_type']]

# 进行数据透视，以 timestamp 和 slug 为基准，把 token_type(Up/Down) 变成列
df_wide = df_trades.pivot_table(
    index=['timestamp', 'slug'], 
    columns='token_type', 
    values=val_cols, 
    aggfunc='first' # 数据已经是1秒聚合的了，直接取 first 即可
)

# 重命名列名 (例如把 ('buy_trade_size', 'Up') 变成 'Up_buy_trade_size')
df_wide.columns = [f"{token}_{metric}" for metric, token in df_wide.columns]
df_wide.reset_index(inplace=True)

# ==========================================
# 4. 缺失值处理与保存
# ==========================================
# 某秒只有 Up 交易没 Down 交易时，Down 的 size 和 count 应该填 0，VWAP 保持 NaN
fill_zero_cols = [c for c in df_wide.columns if any(k in c for k in ['_size', '_count', '_nominal_value'])]
df_wide[fill_zero_cols] = df_wide[fill_zero_cols].fillna(0)

# 保存最终版宽表
df_wide.to_parquet(OUTPUT_PATH)

print(f"\n🎉 转换完成！表结构已完美兼容。")
print(f"📊 新表行数: {df_wide.shape[0]} 行")
print(f"📊 新表列数: {df_wide.shape[1]} 列")
print(f"💾 已保存至: {OUTPUT_PATH}\n")

print("👀 前 5 行数据预览:")
print(df_wide.head())

🚀 正在加载交易总表...
🔍 正在构建 Token 映射字典...
🔄 正在将长表横向展开为宽表 (Pivot)...

🎉 转换完成！表结构已完美兼容。
📊 新表行数: 779234 行
📊 新表列数: 18 列
💾 已保存至: /Volumes/KIOXIA/Project/lob/final_aggregated_trades/grand_total_trades_WIDE_master.parquet

👀 前 5 行数据预览:
                  timestamp                       slug  Down_buy_trade_count  \
0 2025-12-01 05:00:23+00:00  btc-updown-15m-1764565200                   0.0   
1 2025-12-01 05:00:24+00:00  btc-updown-15m-1764565200                   0.0   
2 2025-12-01 05:00:25+00:00  btc-updown-15m-1764565200                   2.0   
3 2025-12-01 05:00:26+00:00  btc-updown-15m-1764565200                   1.0   
4 2025-12-01 05:00:27+00:00  btc-updown-15m-1764565200                   1.0   

   Up_buy_trade_count  Down_buy_trade_size  Up_buy_trade_size  \
0                 1.0                  0.0           2.272726   
1                 1.0                  0.0         111.720000   
2                 0.0                 19.0           0.000000   
3                 2.0                

In [9]:
df_wide

,timestamp,slug,Down_buy_trade_count,Up_buy_trade_count,Down_buy_trade_size,Up_buy_trade_size,Down_buy_trade_vwap,Up_buy_trade_vwap,Down_sell_trade_count,Up_sell_trade_count,Down_sell_trade_size,Up_sell_trade_size,Down_sell_trade_vwap,Up_sell_trade_vwap,Down_taker_buy_trade_nominal_value,Up_taker_buy_trade_nominal_value,Down_taker_sell_trade_nominal_value,Up_taker_sell_trade_nominal_value
0,2025-12-01 05:00:23+00:00,btc-updown-15m-1764565200,0.0,1.0,0.0,2.272726,NaN,0.44,0.0,0.0,0.00,0.0,NaN,NaN,0.00,0.999999,0.0000,0.0
1,2025-12-01 05:00:24+00:00,btc-updown-15m-1764565200,0.0,1.0,0.0,111.720000,NaN,0.45,0.0,0.0,0.00,0.0,NaN,NaN,0.00,50.274000,0.0000,0.0
2,2025-12-01 05:00:25+00:00,btc-updown-15m-1764565200,2.0,0.0,19.0,0.000000,0.564737,NaN,0.0,0.0,0.00,0.0,NaN,NaN,10.73,0.000000,0.0000,0.0
3,2025-12-01 05:00:26+00:00,btc-updown-15m-1764565200,1.0,2.0,9.0,10.000000,0.580000,0.43,0.0,0.0,0.00,0.0,NaN,NaN,5.22,4.300000,0.0000,0.0
4,2025-12-01 05:00:27+00:00,btc-updown-15m-1764565200,1.0,0.0,10.0,0.000000,0.580000,NaN,0.0,0.0,0.00,0.0,NaN,NaN,5.80,0.000000,0.0000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
779229,2025-12-15 05:00:28+00:00,btc-updown-15m-1765773900,0.0,0.0,0.0,0.000000,NaN,NaN,1.0,0.0,249.98,0.0,0.99,NaN,0.00,0.000000,247.4802,0.0
779230,2025-12-15 05:00:29+00:00,btc-updown-15m-1765773900,0.0,0.0,0.0,0.000000,NaN,NaN,1.0,0.0,443.82,0.0,0.99,NaN,0.00,0.000000,439.3818,0.0
779231,2025-12-15 05:00:36+00:00,btc-updown-15m-1765773900,0.0,0.0,0.0,0.000000,NaN,NaN,1.0,0.0,1.63,0.0,0.99,NaN,0.00,0.000000,1.6137,0.0
779232,2025-12-15 05:00:49+00:00,btc-updown-15m-1765773900,0.0,0.0,0.0,0.000000,NaN,NaN,1.0,0.0,1.38,0.0,0.99,NaN,0.00,0.000000,1.3662,0.0


In [6]:
data.shape

(1060114, 59)

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob
import os

# ==========================================
# 1. 基础路径配置
# ==========================================
BASE_DIR = Path('/Volumes/KIOXIA/Project/lob')
INPUT_DIR = BASE_DIR / 'raw_transactions_only'
OUTPUT_DIR = BASE_DIR / 'advanced_hft_features'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEEKS_TO_PROCESS = ['week1', 'week2']

print("🔍 正在构建 Token 映射字典 (asset_id -> Up/Down)...")
mapping_dict = {}
csv_files = glob.glob(str(BASE_DIR / '*' / 'trades_week*.csv'))
for f in csv_files:
    df_meta = pd.read_csv(f, dtype={'token_id': str})
    temp_dict = df_meta.drop_duplicates('token_id').set_index('token_id')['token_name'].to_dict()
    mapping_dict.update(temp_dict)
print(f"   [√] 成功加载 {len(mapping_dict)} 个 Token 映射。")

# ==========================================
# 2. 核心处理函数：重采样 + 因子计算 + 宽表透视
# ==========================================
def process_advanced_features(df_raw, slug):
    if df_raw.empty: return pd.DataFrame()

    df_raw['token_type'] = df_raw['asset_id'].map(mapping_dict)
    df_raw.dropna(subset=['token_type'], inplace=True)
    if df_raw.empty: return pd.DataFrame()

    df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
    df_raw['notional'] = df_raw['price'] * df_raw['size']

    # 基础聚合
    agg_df = df_raw.groupby(['timestamp', 'token_type', 'side']).agg(
        trade_size=('size', 'sum'),
        trade_count=('transaction_hash', 'count'),
        notional=('notional', 'sum')
    ).reset_index()

    # 计算 VWAP (如果有量)
    agg_df['vwap'] = agg_df['notional'] / agg_df['trade_size']
    
    # 组合前缀，用于透视 (例如: Up_BUY)
    agg_df['pivot_key'] = agg_df['token_type'] + '_' + agg_df['side']
    
    val_cols = ['trade_size', 'trade_count', 'notional', 'vwap']
    
    # 透视为宽表
    df_wide = agg_df.pivot(index='timestamp', columns='pivot_key', values=val_cols)
    df_wide.columns = [f"{col[1]}_{col[0]}" for col in df_wide.columns]
    
    # 【非常重要】：对齐到连续的1秒时间网格
    # 因为如果不做这一步，连续10秒没交易，这10秒的行都会直接消失
    full_idx = pd.date_range(start=df_wide.index.min(), end=df_wide.index.max(), freq='1S')
    df_wide = df_wide.reindex(full_idx)
    df_wide.index.name = 'timestamp'
    df_wide.reset_index(inplace=True)
    
    # --------------------------------------------------
    # B. 缺失值精确处理 (听你的，去掉繁琐的，精确填0)
    # --------------------------------------------------
    expected_prefixes = ['Up_BUY', 'Up_SELL', 'Down_BUY', 'Down_SELL']
    
    # 1. 确保所有列存在
    for pfx in expected_prefixes:
        for metric in val_cols:
            col_name = f"{pfx}_{metric}"
            if col_name not in df_wide.columns:
                df_wide[col_name] = np.nan

    # 2. 属于“量”的列，没交易就填 0
    zero_fill_metrics = ['trade_size', 'trade_count', 'notional']
    for pfx in expected_prefixes:
        for metric in zero_fill_metrics:
            df_wide[f"{pfx}_{metric}"] = df_wide[f"{pfx}_{metric}"].fillna(0)

    # 3. 属于“价”的列 (VWAP)，没交易就向前填充 (ffill)
    vwap_cols = [f"{pfx}_vwap" for pfx in expected_prefixes]
    df_wide[vwap_cols] = df_wide[vwap_cols].ffill()
    # 如果开头还有 NaN (比如最开始的几秒没交易)，填0或填盘口中间价都可以，这里简单向后填充
    df_wide[vwap_cols] = df_wide[vwap_cols].bfill()

    # --------------------------------------------------
    # C. 计算最终因子：Net_OFI
    # --------------------------------------------------
    bullish_flow = df_wide['Up_BUY_trade_size'] + df_wide['Down_SELL_trade_size']
    bearish_flow = df_wide['Up_SELL_trade_size'] + df_wide['Down_BUY_trade_size']
    
    # 加上 1e-8 防止分母为 0。如果全没交易，OFI 会是 0（中立）
    df_wide['Net_OFI'] = (bullish_flow - bearish_flow) / (bullish_flow + bearish_flow + 1e-8)

    df_wide['slug'] = slug
    return df_wide

# ==========================================
# 3. 执行循环处理
# ==========================================
all_dfs = []
for week in WEEKS_TO_PROCESS:
    print(f"\n🚀 开始提取 {week.upper()} 的高级特征")
    WEEK_INPUT_DIR = INPUT_DIR / week
    WEEK_OUTPUT_DIR = OUTPUT_DIR / week
    WEEK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    parquet_files = glob.glob(str(WEEK_INPUT_DIR / '*_raw_txs.parquet'))
    
    for file_path in parquet_files:
        slug = os.path.basename(file_path).replace('_raw_txs.parquet', '')
        df_raw = pd.read_parquet(file_path)
        
        df_features = process_advanced_features(df_raw, slug)
        
        if not df_features.empty:
            out_file = WEEK_OUTPUT_DIR / f"{slug}_hft_features.parquet"
            df_features.to_parquet(out_file)
            all_dfs.append(df_features)
            print(f"     [√] 处理完成 -> {slug} (行数: {len(df_features)})")

if all_dfs:
    print("\n🔗 正在拼接最终大总表...")
    df_grand_total = pd.concat(all_dfs, ignore_index=True)
    df_grand_total.sort_values(by=['timestamp', 'slug'], inplace=True)
    df_grand_total.reset_index(drop=True, inplace=True)
    
    grand_output = OUTPUT_DIR / "grand_total_HFT_master.parquet"
    df_grand_total.to_parquet(grand_output)
    print(f"🎉 高频因子大总表生成完毕！保存在: {grand_output}")

🔍 正在构建 Token 映射字典 (asset_id -> Up/Down)...
   [√] 成功加载 2622 个 Token 映射。

🚀 开始提取 WEEK1 的高级特征


/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:36: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:60: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  full_idx = pd.date_range(start=df_wide.index.min(), end=df_wide.index.max(), freq='1S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:36: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:60: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  full_idx = pd.date_range(start=df_wide.index.min(), end=df_wide.inde

     [√] 处理完成 -> btc-updown-15m-1765153800 (行数: 869)
     [√] 处理完成 -> btc-updown-15m-1764838800 (行数: 867)
     [√] 处理完成 -> btc-updown-15m-1764656100 (行数: 856)
     [√] 处理完成 -> btc-updown-15m-1765081800 (行数: 898)
     [√] 处理完成 -> btc-updown-15m-1764792900 (行数: 880)
     [√] 处理完成 -> btc-updown-15m-1764834300 (行数: 878)
     [√] 处理完成 -> btc-updown-15m-1765107900 (行数: 881)
     [√] 处理完成 -> btc-updown-15m-1764567900 (行数: 874)
     [√] 处理完成 -> btc-updown-15m-1764745200 (行数: 879)
     [√] 处理完成 -> btc-updown-15m-1764603000 (行数: 850)
     [√] 处理完成 -> btc-updown-15m-1765127700 (行数: 858)
     [√] 处理完成 -> btc-updown-15m-1765084500 (行数: 912)
     [√] 处理完成 -> btc-updown-15m-1764810000 (行数: 829)
     [√] 处理完成 -> btc-updown-15m-1764843300 (行数: 819)
     [√] 处理完成 -> btc-updown-15m-1764642600 (行数: 881)
     [√] 处理完成 -> btc-updown-15m-1764981900 (行数: 209)
     [√] 处理完成 -> btc-updown-15m-1765103400 (行数: 884)
     [√] 处理完成 -> btc-updown-15m-1764918000 (行数: 884)
     [√] 处理完成 -> btc-updown-15m-1764627300 (行数

/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:36: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:60: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  full_idx = pd.date_range(start=df_wide.index.min(), end=df_wide.index.max(), freq='1S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:36: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:60: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  full_idx = pd.date_range(start=df_wide.index.min(), end=df_wide.inde

     [√] 处理完成 -> btc-updown-15m-1764882000 (行数: 885)
     [√] 处理完成 -> btc-updown-15m-1765010700 (行数: 906)
     [√] 处理完成 -> btc-updown-15m-1764673200 (行数: 887)
     [√] 处理完成 -> btc-updown-15m-1765156500 (行数: 853)
     [√] 处理完成 -> btc-updown-15m-1764751500 (行数: 926)
     [√] 处理完成 -> btc-updown-15m-1764620100 (行数: 865)
     [√] 处理完成 -> btc-updown-15m-1764644400 (行数: 853)
     [√] 处理完成 -> btc-updown-15m-1765118700 (行数: 923)
     [√] 处理完成 -> btc-updown-15m-1764845100 (行数: 890)
     [√] 处理完成 -> btc-updown-15m-1764966600 (行数: 896)
     [√] 处理完成 -> btc-updown-15m-1764578700 (行数: 557)
     [√] 处理完成 -> btc-updown-15m-1765082700 (行数: 881)
     [√] 处理完成 -> btc-updown-15m-1764817200 (行数: 847)
     [√] 处理完成 -> btc-updown-15m-1764946800 (行数: 847)
     [√] 处理完成 -> btc-updown-15m-1765068300 (行数: 861)
     [√] 处理完成 -> btc-updown-15m-1764815400 (行数: 893)
     [√] 处理完成 -> btc-updown-15m-1765018800 (行数: 892)
     [√] 处理完成 -> btc-updown-15m-1764779400 (行数: 899)
     [√] 处理完成 -> btc-updown-15m-1764792000 (行数

...

     [√] 处理完成 -> btc-updown-15m-1765694700 (行数: 878)
     [√] 处理完成 -> btc-updown-15m-1765568700 (行数: 914)
     [√] 处理完成 -> btc-updown-15m-1765440900 (行数: 745)
     [√] 处理完成 -> btc-updown-15m-1765457100 (行数: 862)
     [√] 处理完成 -> btc-updown-15m-1765352700 (行数: 841)
     [√] 处理完成 -> btc-updown-15m-1765713600 (行数: 894)
     [√] 处理完成 -> btc-updown-15m-1765663200 (行数: 875)
     [√] 处理完成 -> btc-updown-15m-1765741500 (行数: 891)
     [√] 处理完成 -> btc-updown-15m-1765460700 (行数: 919)
     [√] 处理完成 -> btc-updown-15m-1765233900 (行数: 918)
     [√] 处理完成 -> btc-updown-15m-1765414800 (行数: 826)
     [√] 处理完成 -> btc-updown-15m-1765665000 (行数: 886)
     [√] 处理完成 -> btc-updown-15m-1765715400 (行数: 861)
     [√] 处理完成 -> btc-updown-15m-1765434600 (行数: 826)
     [√] 处理完成 -> btc-updown-15m-1765276200 (行数: 855)
     [√] 处理完成 -> btc-updown-15m-1765354500 (行数: 533)
     [√] 处理完成 -> btc-updown-15m-1765280700 (行数: 831)
     [√] 处理完成 -> btc-updown-15m-1765593900 (行数: 812)
     [√] 处理完成 -> btc-updown-15m-1765486800 (行数

/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:36: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:60: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  full_idx = pd.date_range(start=df_wide.index.min(), end=df_wide.index.max(), freq='1S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:36: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:60: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  full_idx = pd.date_range(start=df_wide.index.min(), end=df_wide.inde

     [√] 处理完成 -> btc-updown-15m-1765443600 (行数: 847)
     [√] 处理完成 -> btc-updown-15m-1765351800 (行数: 859)
     [√] 处理完成 -> btc-updown-15m-1765710900 (行数: 861)
     [√] 处理完成 -> btc-updown-15m-1765529100 (行数: 829)
     [√] 处理完成 -> btc-updown-15m-1765608300 (行数: 909)
     [√] 处理完成 -> btc-updown-15m-1765249200 (行数: 888)
     [√] 处理完成 -> btc-updown-15m-1765646100 (行数: 878)
     [√] 处理完成 -> btc-updown-15m-1765325700 (行数: 875)
     [√] 处理完成 -> btc-updown-15m-1765421100 (行数: 841)
     [√] 处理完成 -> btc-updown-15m-1765445400 (行数: 857)
     [√] 处理完成 -> btc-updown-15m-1765206000 (行数: 862)
     [√] 处理完成 -> btc-updown-15m-1765305900 (行数: 905)
     [√] 处理完成 -> btc-updown-15m-1765313100 (行数: 852)

🔗 正在拼接最终大总表...


/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:36: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:60: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  full_idx = pd.date_range(start=df_wide.index.min(), end=df_wide.index.max(), freq='1S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:36: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_raw['timestamp'] = df_raw['timestamp'].dt.floor('S')
/var/folders/j4/kwkcy_yx4s5fg992bwd59h140000gn/T/ipykernel_45690/1266925292.py:60: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  full_idx = pd.date_range(start=df_wide.index.min(), end=df_wide.inde

🎉 高频因子大总表生成完毕！保存在: /Volumes/KIOXIA/Project/lob/advanced_hft_features/grand_total_HFT_master.parquet


In [14]:
df_grand_total

,timestamp,Down_BUY_trade_size,Down_SELL_trade_size,Up_BUY_trade_size,Up_SELL_trade_size,Down_BUY_trade_count,Down_SELL_trade_count,Up_BUY_trade_count,Up_SELL_trade_count,Down_BUY_notional,Down_SELL_notional,Up_BUY_notional,Up_SELL_notional,Down_BUY_vwap,Down_SELL_vwap,Up_BUY_vwap,Up_SELL_vwap,Net_OFI,slug
0,2025-12-01 05:00:23+00:00,0.0,0.00,2.272726,0.0,0.0,0.0,1.0,0.0,0.00,0.0000,0.999999,0.0,0.564737,0.57,0.44,0.41,1.000000,btc-updown-15m-1764565200
1,2025-12-01 05:00:24+00:00,0.0,0.00,111.720000,0.0,0.0,0.0,1.0,0.0,0.00,0.0000,50.274000,0.0,0.564737,0.57,0.45,0.41,1.000000,btc-updown-15m-1764565200
2,2025-12-01 05:00:25+00:00,19.0,0.00,0.000000,0.0,2.0,0.0,0.0,0.0,10.73,0.0000,0.000000,0.0,0.564737,0.57,0.45,0.41,-1.000000,btc-updown-15m-1764565200
3,2025-12-01 05:00:26+00:00,9.0,0.00,10.000000,0.0,1.0,0.0,2.0,0.0,5.22,0.0000,4.300000,0.0,0.580000,0.57,0.43,0.41,0.052632,btc-updown-15m-1764565200
4,2025-12-01 05:00:27+00:00,10.0,0.00,0.000000,0.0,1.0,0.0,0.0,0.0,5.80,0.0000,0.000000,0.0,0.580000,0.57,0.43,0.41,-1.000000,btc-updown-15m-1764565200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1050777,2025-12-15 05:00:58+00:00,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.00,0.0000,0.000000,0.0,0.982442,0.99,0.01,0.05,0.000000,btc-updown-15m-1765773900
1050778,2025-12-15 05:00:59+00:00,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.00,0.0000,0.000000,0.0,0.982442,0.99,0.01,0.05,0.000000,btc-updown-15m-1765773900
1050779,2025-12-15 05:01:00+00:00,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.00,0.0000,0.000000,0.0,0.982442,0.99,0.01,0.05,0.000000,btc-updown-15m-1765773900
1050780,2025-12-15 05:01:01+00:00,0.0,0.00,0.000000,0.0,0.0,0.0,0.0,0.0,0.00,0.0000,0.000000,0.0,0.982442,0.99,0.01,0.05,0.000000,btc-updown-15m-1765773900
